# TwinFit — IDM-VTON on the AMD Hackathon Notebook
Run cells top to bottom. **Do not stop the instance while working — GPU slots are shared and get claimed instantly.**

At the end you get a public `*.gradio.live` URL → paste it into `backend/.env` on your Mac as `TRYON_HF_SPACE=<url>` and restart the backend. Try-ons then run on this AMD GPU.

In [ ]:
# 1. Verify the AMD GPU is visible
import torch
print("torch:", torch.__version__)
print("gpu available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — grab a GPU slot first")

In [ ]:
# 2. Clone IDM-VTON + install deps (~5 min). Torch is already installed for ROCm — don't touch it.
%cd ~
!git clone https://github.com/yisol/IDM-VTON.git 2>/dev/null || echo 'already cloned'
%cd ~/IDM-VTON
!grep -viE '^(torch|torchvision|torchaudio)' requirements.txt > req_no_torch.txt
!pip install -q -r req_no_torch.txt
!pip install -q ninja gradio huggingface_hub
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
# 3. Preprocessing checkpoints (~1 min)
import os, shutil, urllib.request
from huggingface_hub import hf_hub_download
os.makedirs('ckpt/densepose', exist_ok=True)
os.makedirs('ckpt/humanparsing', exist_ok=True)
os.makedirs('ckpt/openpose/ckpts', exist_ok=True)

dp = 'ckpt/densepose/model_final_162be9.pkl'
if not os.path.exists(dp):
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl', dp)

for f in ['ckpt/humanparsing/parsing_atr.onnx',
          'ckpt/humanparsing/parsing_lip.onnx',
          'ckpt/openpose/ckpts/body_pose_model.pth']:
    if not os.path.exists(f):
        p = hf_hub_download(repo_id='yisol/IDM-VTON', repo_type='space', filename=f)
        shutil.copy(p, f)
print('checkpoints ready')

In [ ]:
# 4. Patch the demo to expose a public URL
!sed -i 's/image_blocks.launch()/image_blocks.launch(share=True)/' gradio_demo/app.py
!grep 'launch' gradio_demo/app.py

In [ ]:
# 5. LAUNCH (first run downloads ~15GB of weights — 10-20 min; keep this cell running!)
# When you see:  Running on public URL: https://xxxx.gradio.live
# → on your Mac, set in backend/.env:   TRYON_HF_SPACE=https://xxxx.gradio.live
# → restart backend. Done: try-ons now run on THIS AMD GPU.
%cd ~/IDM-VTON/gradio_demo
!python app.py